### μ and φ are unknowns

Next step: make μ and φ unknown and estimate them with partial pooling across genes, instead of fixing them. 

```Bash
          exact      bayes
gene j:     μ  -->   q_j  (m_q, σ_q)            gene mean expression  
            φ  -->   φ_j  (a0 + a1/q_j), σ_φ    dispersion around a mean trend  
```

#### Log-normal

```Bash
gene j:     log q_j  ~ Normal(m_q, σ_q)                       gene mean expression
            log φ_j  ~ Normal(log(a0 + a1/q_j), σ_φ)          dispersion around a mean trend
```

#### Build a reusable script that simulates the count matrix, fits a partially pooled negative binomial model with NUTS

Environment **bayes** is set up.

Build a reusable script that simulates the count matrix, fits a partially pooled negative binomial model with NUTS, and compares it against unpooled per-gene estimates.

The fit converged cleanly, except the population-mean parameter m_q mixed slowly (ESS 118, r̂ 1.06). Therefore, I'm switching log q to a centred parameterisation and refitting.

Mixing is now good for every parameter (m_q ESS went from 118 to 7113, and all r̂ = 1.00). 

Below, the plot the shrinkage and the parameter recovery.

Panels a and b have text sitting over data points, and panel c's key covers the σ_φ row. The panel b annotation also wrongly says the MLE goes to φ → ∞ when only the lower bound is reached, so I'm fixing all three.

I built a hierarchical negative binomial model in which each gene has its own mean and dispersion, and those values are drawn from a shared population distribution instead of being fixed. In a simulation it recovered the true gene dispersions about 7× more accurately than fitting each gene separately: the RMSE on log φ was 0.39 versus 2.80.

![Hierarchical vs per-gene dispersion estimates](../figures/hierarchical_estimates.jpg)

### 1. The model

In the previous simulation μ and φ were fixed numbers. Here they get their own distributions, organised in three levels:

```Bash

population (shared):  m_q, σ_q, a0, a1, σ_φ  
        │  
        v  
gene j:     log q_j  ~ Normal(m_q, σ_q)                       gene mean expression  
            log φ_j  ~ Normal(log(a0 + a1/q_j), σ_φ)          dispersion around a mean trend  
        │  
        v  
cell i, gene j:  
            λ_ij ~ Gamma(scale φ_j * s_i * q_j, shape 1/φ_j)      biological rate (as before)  
            y_ij ~ Poisson(λ_ij)  ==>  y_ij ~ NB(s_i·q_j, φ_j)  
  
            where, s_i is the size factors  

```

- **The Gamma layer stays the same.** 

  - It is integrated out analytically, which gives the NB likelihood.

  - NUTS (the sampler) therefore only has to explore about 600 parameters instead of 12,000 latent λ values.

- **The trend a0 + a1/q is the parametric dispersion trend used in DESeq2.**

  - At low mean, dispersion is dominated by the a1/q term; at high mean it levels off at a0.

- **σ_φ sets how strongly genes are pooled, and it is estimated from the data.** 
  - If σ_φ comes out small, each gene's φ is pulled firmly onto the trend; 

  - if large, genes keep their own values.

- **The size factors s_i are treated as known.** 
  - In a real pipeline they would come from library size.

<br>

### **NumPyro**

Probabilistic programming powered by JAX for autograd and JIT compilation to GPU/TPU/CPU

NumPyro is a lightweight probabilistic programming library that provides a NumPy backend for Pyro. We rely on JAX for automatic differentiation and JIT compilation to GPU / CPU. NumPyro is under active development, so beware of brittleness, bugs, and changes to the API as the design evolves.

Website: https://numpyro.com/

Tutorial: https://num.pyro.ai/en/stable/

Github: https://github.com/pyro-ppl/numpyro


#### The core NumPyro code is about 15 lines:

See the method **model** in hiernb_model.py.


#### **Parameterisation affects sampling.** 

Gene means are pinned down tightly by the data, so they work best sampled directly ("centred"). Dispersions are poorly determined with 40 cells, so they work best sampled as an offset from the trend ("non-centred"). My first run sampled both as offsets, and the population mean m_q mixed badly: effective sample size (ESS) 118, r̂ 1.06. Sampling log q directly raised that ESS to 7113 with r̂ 1.00.


### 2. Simulation results

I simulated 300 genes × 40 cells; 37.1% of counts are zero. I fitted 4 chains with 1000 warm-up and 1000 sampling iterations each, which took 1.6 s on the CPU.

**Sampler diagnostics:** 0 divergences, r̂ ≤ 1.01 for every parameter, and minimum gene-level ESS 3760.

**Hyperparameters (panel c):**

| Parameter | True value | Posterior mean [90% interval] |
|---|---|---|
| a0 | 0.15 | 0.120 [0.095, 0.143] |
| a1 | 0.5 | 0.473 [0.396, 0.554] |
| σ_φ | 0.5 | 0.501 [0.419, 0.586] |
| σ_q | 1.5 | 1.382 [1.290, 1.480] |

For a0 and σ_q the true value falls outside the 90% interval. This is a property of this particular random draw, not a fitting error. Refitting the trend by least squares directly to the 300 true φ values gives a0 = 0.126. The standard deviation of the 300 simulated log q values is 1.391, close to the posterior mean of 1.38.

#### **Gene dispersions (panels a and b):**

- **Per-gene MLE (no pooling):** 33 of 300 genes get an estimated φ < 0.01. Of those, 10 hit the optimiser's lower bound of 10⁻⁴, meaning the fit treats the gene as essentially Poisson.

- **Hierarchical model:** these genes are pulled towards the trend.

| Estimator | RMSE log φ, all 300 genes | RMSE log φ, 107 genes with mean < 1 |
|---|---|---|
| Per-gene MLE (no pooling) | 2.80 | 4.10 |
| Hierarchical | 0.39 | 0.51 |

The 90% posterior intervals contain the true φ for 93.7% of genes, so the uncertainty is calibrated.

This is the same reason DESeq2, edgeR and glmGamPoi shrink dispersions towards a fitted trend. Those tools use an empirical-Bayes shortcut: estimate the trend and σ_φ once, then shrink each gene. The df_full Bayesian model also carries the uncertainty in the trend into each gene's estimate.

### 3. Applying it to the data

1. **Input:** use a raw count matrix (cells × genes), not normalised values. Set s_i from library size divided by its geometric mean.

2. **Scale:** for more than about 2000 genes or 10k cells, replace **MCMC(NUTS…)** with **SVI** and an **AutoNormal** guide. The model code stays the same, and minibatches of cells can be drawn with **numpyro.plate("cells", CELLS, subsample_size=…)**.

3. **Possible extensions:**
   - **Condition effects:** log μ_ij = log s_i + β_j0 + β_j1·x_i, with β_j1 ~ Normal(0, τ). This gives differential expression with shrinkage of the fold changes.
   - **Per cell type:** one trend per cell type, with a parent level above them.
   - **Zero inflation:** use **ZeroInflatedNegativeBinomial2** if posterior predictive checks show more zeros than the NB predicts.

Files are in the output tray:

- hier_nb_model.py — df_full script: simulation, no-pooling MLE, NUTS fit, diagnostics

- hier_nb_hyperparameters.csv — hyperparameter posterior summary
- hier_nb_shrinkage.png — the figure above

### JAX

JAX is a high-performance, numerical computing library developed by Google that combines a NumPy-like API with flexible program transformations. Unlike monolithic frameworks like PyTorch or TensorFlow, JAX is designed as a minimalist tool for transforming Python functions.

Quickstart: https://docs.jax.dev/en/latest/quickstart.html

Youtube: 

- https://www.youtube.com/watch?v=wq-UsiOkBRU
- https://www.youtube.com/watch?v=SstuvS-tVc0
- https://www.youtube.com/watch?v=uySOfXq-II0


In [ ]:
import os, sys
from pathlib import Path

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=4"

import time, json
import numpy as np
import jax, jax.numpy as jnp
import numpyro, numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from scipy import optimize, special
import arviz as az

from hier_nb_model import *

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.Basic import pdwritecsv, pdreadcsv, create_dir


```Bash
pip install -U "jax[cuda12]" numpyro arviz
python -c "import jax; print(jax.devices())"   # should list CudaDevice(id=0)
```

In [ ]:
root_source = create_dir(ROOT0, 'notebooks_multiproject')

root_figure = create_dir(root_source, 'figures')
root_result = create_dir(root_source, 'results')

### New random generator

see: https://numpy.org/devdocs//reference/random/generator.html

In [ ]:
numpyro.set_host_device_count(4)

rng = np.random.default_rng(7)

### simulation

In [ ]:
GENES = 300
CELLS = 40                      # genes, cells (few cells -> pooling matters)

dic_params = dict(a0=0.15, a1=0.5, sigma_phi=0.5, m_q=0.5, sigma_q=1.5)

In [ ]:

s = rng.lognormal(0, 0.4, CELLS)
s /= np.exp(np.log(s).mean())       # geometric mean 1

log_q = rng.normal(dic_params["m_q"], dic_params["sigma_q"], GENES)
q = np.exp(log_q)

phi_trend = dic_params["a0"] + dic_params["a1"] / q
phi = np.exp(rng.normal(np.log(phi_trend), dic_params["sigma_phi"]))

mu = s[:, None] * q[None, :]                        # CELLS x GENES

rand_gamma = rng.gamma(shape=1/phi, scale=phi*mu)   # explicit Gamma layer
rand_poi = rng.poisson(rand_gamma)

print(f"rand_poi (Y): {CELLS} cells x {GENES} genes, fraction zeros = {np.mean(rand_poi == 0):.3f}")

mle_phi, mle_q = np.empty(GENES), np.empty(GENES)
for j in range(GENES):
    y = rand_poi[:, j]
    x0 = [np.log(max(y.mean(), 1e-3)), 0.0]
    res = optimize.minimize(nb_negll, x0, args=(y, s), method="L-BFGS-B",
                            bounds=[(-10, 10), (np.log(1e-4), np.log(1e3))])
    mle_q[j], mle_phi[j] = np.exp(res.x)


In [ ]:

t0 = time.time()

mcmc = MCMC(NUTS(model, target_accept_prob=0.9), num_warmup=1000, num_samples=1000,
            num_chains=4, chain_method="parallel", progress_bar=False)

mcmc.run(jax.random.PRNGKey(0), s=jnp.array(s), rand_poi=jnp.array(rand_poi), GENES=GENES)

idata = az.from_numpyro(mcmc)

print(f"NUTS wall time: {time.time() - t0:.1f} s")

In [ ]:
hyp = ["a0", "a1", "sigma_phi", "m_q", "sigma_q"]
df_summ = az.summary(idata, var_names=hyp, ci_prob=0.9, ci_kind="hdi", round_to="none")
df_summ.filter(regex=r"^(mean|hdi.*|r_hat|ess_bulk)$").round(3)

In [ ]:
df_full = az.summary(idata, var_names=["log_phi", "log_q"], kind="diagnostics", round_to="none")
print(f"max r_hat (gene-level) = {df_full['r_hat'].max():.3f}, "
      f"min ess_bulk = {df_full['ess_bulk'].min():.0f}, "
      f"divergences = {int(mcmc.get_extra_fields()['diverging'].sum())}")

print(df_full.shape)
df_full.head(3)

In [ ]:
t0 = time.time()

post = mcmc.get_samples()
hb_phi = np.exp(np.median(np.asarray(post["log_phi"]), axis=0))
hb_lo, hb_hi = np.exp(np.percentile(np.asarray(post["log_phi"]), [5, 95], axis=0))
cover = np.mean((phi >= hb_lo) & (phi <= hb_hi))

rmse = lambda est: np.sqrt(np.mean((np.log(est) - np.log(phi)) ** 2))
low = q < 1
out = dict(
    rmse_log_phi_mle=rmse(mle_phi), rmse_log_phi_hb=rmse(hb_phi),
    rmse_low_mean_mle=float(np.sqrt(np.mean((np.log(mle_phi[low]) - np.log(phi[low])) ** 2))),
    rmse_low_mean_hb=float(np.sqrt(np.mean((np.log(hb_phi[low]) - np.log(phi[low])) ** 2))),
    n_low=int(low.sum()), coverage_90=float(cover),
    mle_at_lower_bound=int(np.sum(mle_phi <= 1.01e-4)),
)
print(json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in out.items()}))

np.savez("hier_nb_results.npz", q=q, phi=phi, mle_phi=mle_phi, mle_q=mle_q, hb_phi=hb_phi,
         hb_lo=hb_lo, hb_hi=hb_hi, s=s, rand_poi=rand_poi,
         a0=np.asarray(post["a0"]), a1=np.asarray(post["a1"]),
         sigma_phi=np.asarray(post["sigma_phi"]))

print(f"Posterior wall time: {time.time() - t0:.1f} s")

filename = root_result / "hier_nb_hyperparameters.csv"
df_summ.to_csv(filename)

In [ ]:
df_summ